In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import re
import sys
from credit_default_risk.config import TARGET_COL, ID_COL, PROCESSED_TRAIN_PATH, PROCESSED_TEST_PATH
from IPython.display import Markdown, display
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from credit_default_risk.features import prepare_catboost_frame
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [2]:
train_df = pd.read_parquet(PROCESSED_TRAIN_PATH)
print("Train:", train_df.shape)
display(train_df.head())

Train: (246008, 99)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,NUM_DOCUMENTS,BUREAU_NUM_CREDITS,BUREAU_ACTIVE_CREDIT_COUNT,BUREAU_MAX_DAYS_OVERDUE,PREV_APPLICATION_COUNT,PREV_APPROVED_COUNT,PREV_APPROVED_REQUESTED_AMT_SUM,PREV_APPROVED_GRANTED_AMT_SUM,PREV_APPROVED_REQUESTED_AMT_MEAN,PREV_APPROVED_GRANTED_AMT_MEAN,PREV_APPROVED_REQUESTED_TO_GRANTED_RATIO_MEAN,PREV_APPROVAL_RATE,CREDIT_CARD_AVG_UTILIZATION,CREDIT_CARD_AVG_BALANCE,INSTALLMENT_LATE_PAYMENT_RATE,INSTALLMENT_PAYMENT_TO_DUE_RATIO,AGE_YEARS,EMPLOYMENT_YEARS,AGE_BIN,EMPLOYMENT_BIN,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,GOODS_TO_CREDIT_RATIO,CREDIT_TO_ANNUITY_RATIO,INCOME_PER_FAMILY_MEMBER
0,310536,0,Cash loans,F,0,0,2,90000.0,227520.0,13189.5,180000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.008230,-12298,-946.0,-6378.0,-4670,NaN,1,1,0,1,0,0,Laborers,4.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 2,0.384582,0.289573,0.622922,0.2248,0.1613,0.9851,0.7987,0.0740,0.32,0.2759,0.3333,0.3750,0.0661,0.1847,0.2322,0.0,0.0,reg oper account,block of flats,0.2197,Panel,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,5.0,2.0,0.0,5.0,5.0,222970.5,693369.0,44594.100,138673.800,0.783398,1.000000,0.346744,213473.232857,0.0,1.060675,33.670089,2.590007,25-34,1-2,2.528000,0.146550,0.791139,17.250085,22500.0
1,365516,0,Cash loans,M,1,1,0,90000.0,161730.0,13095.0,135000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.003069,-15375,-92.0,-1292.0,-1994,23.0,1,1,0,1,0,0,Drivers,2.0,3,3,TUESDAY,13,0,0,0,0,1,1,Other,NaN,0.514261,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,2.0,0.0,2.0,0.0,-509.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,6.0,4.0,289156.5,288688.5,72289.125,72172.125,0.970604,0.666667,NaN,NaN,0.0,1.000000,42.094456,0.251882,35-44,<1,1.797000,0.145500,0.834725,12.350515,45000.0
2,242055,1,Cash loans,M,0,1,0,135000.0,728847.0,26307.0,553500.0,"Spouse, partner",Working,Secondary / secondary special,Single / not married,House / apartment,0.020713,-19307,-1646.0,-7951.0,-2842,NaN,1,1,0,1,1,0,Laborers,1.0,3,2,TUESDAY,5,0,0,0,0,0,0,Business Entity Type 3,NaN,0.486906,0.598926,0.1499,0.1124,0.9891,0.8524,0.0386,0.16,0.1379,0.3750,0.4167,0.3117,0.1231,0.1643,0.0,0.0,reg oper spec account,block of flats,0.1269,Panel,0,1.0,0.0,1.0,0.0,-1474.0,0.0,0.0,2.0,0.0,0.0,2.0,1.0,4.0,2.0,0.0,1.0,1.0,49045.5,45598.5,49045.500,45598.500,1.075595,1.000000,NaN,NaN,0.0,1.000000,52.859685,4.506502,45-54,3-4,5.398867,0.194867,0.759419,27.705440,135000.0
3,454894,1,Cash loans,M,0,0,0,135000.0,474183.0,34636.5,391500.0,Unaccompanied,Commercial associate,Secondary / secondary special,

### Features Selection

In [4]:
X_train = train_df.drop(columns=[TARGET_COL, ID_COL])
y_train = train_df[TARGET_COL]
X_train_cb = prepare_catboost_frame(X_train)

In [5]:
X_train_cb.head()

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,NUM_DOCUMENTS,BUREAU_NUM_CREDITS,BUREAU_ACTIVE_CREDIT_COUNT,BUREAU_MAX_DAYS_OVERDUE,PREV_APPLICATION_COUNT,PREV_APPROVED_COUNT,PREV_APPROVED_REQUESTED_AMT_SUM,PREV_APPROVED_GRANTED_AMT_SUM,PREV_APPROVED_REQUESTED_AMT_MEAN,PREV_APPROVED_GRANTED_AMT_MEAN,PREV_APPROVED_REQUESTED_TO_GRANTED_RATIO_MEAN,PREV_APPROVAL_RATE,CREDIT_CARD_AVG_UTILIZATION,CREDIT_CARD_AVG_BALANCE,INSTALLMENT_LATE_PAYMENT_RATE,INSTALLMENT_PAYMENT_TO_DUE_RATIO,AGE_YEARS,EMPLOYMENT_YEARS,AGE_BIN,EMPLOYMENT_BIN,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,GOODS_TO_CREDIT_RATIO,CREDIT_TO_ANNUITY_RATIO,INCOME_PER_FAMILY_MEMBER
0,Cash loans,F,0.0,0.0,2,90000.0,227520.0,13189.5,180000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.008230,-12298,-946.0,-6378.0,-4670,NaN,1,1,0,1,0,0,Laborers,4.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 2,0.384582,0.289573,0.622922,0.2248,0.1613,0.9851,0.7987,0.0740,0.32,0.2759,0.3333,0.3750,0.0661,0.1847,0.2322,0.0,0.0,reg oper account,block of flats,0.2197,Panel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,5.0,2.0,0.0,5.0,5.0,222970.5,693369.0,44594.100,138673.800,0.783398,1.000000,0.346744,213473.232857,0.0,1.060675,33.670089,2.590007,25-34,1-2,2.528000,0.146550,0.791139,17.250085,22500.0
1,Cash loans,M,1.0,1.0,0,90000.0,161730.0,13095.0,135000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.003069,-15375,-92.0,-1292.0,-1994,23.0,1,1,0,1,0,0,Drivers,2.0,3,3,TUESDAY,13,0,0,0,0,1,1,Other,NaN,0.514261,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,__MISSING__,__MISSING__,NaN,__MISSING__,NaN,2.0,0.0,2.0,0.0,-509.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,6.0,4.0,289156.5,288688.5,72289.125,72172.125,0.970604,0.666667,NaN,NaN,0.0,1.000000,42.094456,0.251882,35-44,<1,1.797000,0.145500,0.834725,12.350515,45000.0
2,Cash loans,M,0.0,1.0,0,135000.0,728847.0,26307.0,553500.0,"Spouse, partner",Working,Secondary / secondary special,Single / not married,House / apartment,0.020713,-19307,-1646.0,-7951.0,-2842,NaN,1,1,0,1,1,0,Laborers,1.0,3,2,TUESDAY,5,0,0,0,0,0,0,Business Entity Type 3,NaN,0.486906,0.598926,0.1499,0.1124,0.9891,0.8524,0.0386,0.16,0.1379,0.3750,0.4167,0.3117,0.1231,0.1643,0.0,0.0,reg oper spec account,block of flats,0.1269,Panel,0.0,1.0,0.0,1.0,0.0,-1474.0,0.0,0.0,2.0,0.0,0.0,2.0,1.0,4.0,2.0,0.0,1.0,1.0,49045.5,45598.5,49045.500,45598.500,1.075595,1.000000,NaN,NaN,0.0,1.000000,52.859685,4.506502,45-54,3-4,5.398867,0.194867,0.759419,27.705440,135000.0
3,Cash loans,M,0.0,0.0,0,135000.0,474183.0,34636.5,391500.0,Unaccompanied,Commercial associate,Secondary / secondary special,Single / no

In [6]:
X_dev, X_val, y_dev, y_val = train_test_split(
    X_train_cb,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)
cat_cols = X_dev.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()


In [9]:
baseline_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02,
    depth=6,
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=100
)
baseline_model.fit(
    X_dev,
    y_dev,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    use_best_model=True
)


0:	test: 0.6761181	best: 0.6761181 (0)	total: 82.7ms	remaining: 2m 45s
100:	test: 0.7456077	best: 0.7456077 (100)	total: 7.63s	remaining: 2m 23s
200:	test: 0.7553801	best: 0.7553801 (200)	total: 15.3s	remaining: 2m 17s
300:	test: 0.7606445	best: 0.7606445 (300)	total: 23.4s	remaining: 2m 11s
400:	test: 0.7638666	best: 0.7638666 (400)	total: 31.2s	remaining: 2m 4s
500:	test: 0.7660694	best: 0.7660694 (500)	total: 39.3s	remaining: 1m 57s
600:	test: 0.7680207	best: 0.7680207 (600)	total: 47.3s	remaining: 1m 50s
700:	test: 0.7696842	best: 0.7696842 (700)	total: 56.3s	remaining: 1m 44s
800:	test: 0.7706148	best: 0.7706164 (799)	total: 1m 5s	remaining: 1m 37s
900:	test: 0.7716351	best: 0.7716392 (899)	total: 1m 13s	remaining: 1m 29s
1000:	test: 0.7724620	best: 0.7724620 (1000)	total: 1m 21s	remaining: 1m 21s
1100:	test: 0.7731182	best: 0.7731216 (1099)	total: 1m 29s	remaining: 1m 13s
1200:	test: 0.7735660	best: 0.7735714 (1199)	total: 1m 37s	remaining: 1m 4s
1300:	test: 0.7739882	best: 0.773

CatBoostClassifier(depth=6, early_stopping_rounds=100, eval_metric='AUC', iterations=2000, learning_rate=0.02, random_seed=42, verbose=100)

In [15]:
val_prob = baseline_model.predict_proba(X_val)[:, 1]

In [16]:
baseline_ap = average_precision_score(y_val, val_prob)
baseline_auc = roc_auc_score(y_val, val_prob)

print(f"Validation AP:  {baseline_ap:.5f}")
print(f"Validation AUC: {baseline_auc:.5f}")

Validation AP:  0.25713
Validation AUC: 0.77587


### Catboost Classifier Feature Importance

In [17]:
importance_df = (
    pd.DataFrame({
        "feature": X_dev.columns,
        "importance": baseline_model.feature_importances_,
    })
    .sort_values("importance", ascending=True)
    .reset_index(drop=True)
)

display(importance_df)

,feature,importance
0,FLAG_MOBIL,0.000000
1,FLAG_CONT_MOBILE,0.002533
2,LIVE_REGION_NOT_WORK_REGION,0.005175
3,REG_REGION_NOT_WORK_REGION,0.019004
4,FLAG_EMAIL,0.024964
5,FLAG_EMP_PHONE,0.034779
6,FONDKAPREMONT_MODE,0.034964
7,FLAG_PHONE,0.039317
8,AMT_REQ_CREDIT_BUREAU_WEEK,0.040238
9,EMERGENCYSTATE_MODE,0.054781


### Permutation Importance

In [18]:
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    baseline_model,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=1,
)

permutation_df = (
    pd.DataFrame({
        "feature": X_val.columns,
        "ap_drop_mean": permutation_result.importances_mean,
        "ap_drop_std": permutation_result.importances_std,
    })
    .sort_values("ap_drop_mean")
    .reset_index(drop=True)
)

display(permutation_df)

,feature,ap_drop_mean,ap_drop_std
0,NAME_INCOME_TYPE,-4.380759e-04,0.000340
1,INCOME_PER_FAMILY_MEMBER,-4.179590e-04,0.000205
2,DEF_60_CNT_SOCIAL_CIRCLE,-2.062459e-04,0.000196
3,DAYS_LAST_PHONE_CHANGE,-1.855783e-04,0.000342
4,PREV_APPROVED_GRANTED_AMT_MEAN,-1.766241e-04,0.000156
5,PREV_APPLICATION_COUNT,-1.575530e-04,0.000259
6,LIVINGAPARTMENTS_MEDI,-1.414180e-04,0.000066
7,WEEKDAY_APPR_PROCESS_START,-1.299847e-04,0.000109
8,LANDAREA_MEDI,-1.079552e-04,0.000035
9,BUREAU_MAX_DAYS_OVERDUE,-9.677432e-05,0.000095


In [19]:
candidate_drop = permutation_df.loc[
    permutation_df["ap_drop_mean"] <= 0,
    "feature",
].tolist()

print("Candidate features:", len(candidate_drop))
print(candidate_drop)

Candidate features: 30
['NAME_INCOME_TYPE', 'INCOME_PER_FAMILY_MEMBER', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE', 'PREV_APPROVED_GRANTED_AMT_MEAN', 'PREV_APPLICATION_COUNT', 'LIVINGAPARTMENTS_MEDI', 'WEEKDAY_APPR_PROCESS_START', 'LANDAREA_MEDI', 'BUREAU_MAX_DAYS_OVERDUE', 'EMERGENCYSTATE_MODE', 'HOUR_APPR_PROCESS_START', 'LIVINGAREA_MEDI', 'REG_REGION_NOT_LIVE_REGION', 'TOTALAREA_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'AMT_REQ_CREDIT_BUREAU_DAY', 'ELEVATORS_MEDI', 'INSTALLMENT_PAYMENT_TO_DUE_RATIO', 'YEARS_BUILD_MEDI', 'AMT_INCOME_TOTAL', 'REG_CITY_NOT_WORK_CITY', 'WALLSMATERIAL_MODE', 'LIVE_REGION_NOT_WORK_REGION', 'HOUSETYPE_MODE', 'FLAG_EMP_PHONE', 'ENTRANCES_MEDI', 'FONDKAPREMONT_MODE', 'NONLIVINGAREA_MEDI', 'FLAG_MOBIL']


In [21]:
redundant_drop = [
    "DAYS_BIRTH",
    "AGE_BIN",
    "DAYS_EMPLOYED",
    "EMPLOYMENT_BIN",
]
X_dev_reduced = X_dev.drop(columns=candidate_drop+redundant_drop)
X_val_reduced = X_val.drop(columns=candidate_drop+redundant_drop)

reduced_cat_cols = [
    col for col in cat_cols
    if col in X_dev_reduced.columns
]

reduced_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    early_stopping_rounds=150,
)

reduced_model.fit(
    X_dev_reduced,
    y_dev,
    cat_features=reduced_cat_cols,
    eval_set=(X_val_reduced, y_val),
    use_best_model=True,
)

reduced_prob = reduced_model.predict_proba(X_val_reduced)[:, 1]

print(f"Baseline AP: {baseline_ap:.5f}")
print(f"Reduced AP:  {average_precision_score(y_val, reduced_prob):.5f}")
print(f"Baseline AUC: {baseline_auc:.5f}")
print(f"Reduced AUC:  {roc_auc_score(y_val, reduced_prob):.5f}")
print(f"Features removed: {len(candidate_drop)}")

0:	test: 0.5818295	best: 0.5818295 (0)	total: 70.4ms	remaining: 3m 31s
200:	test: 0.7609104	best: 0.7609104 (200)	total: 10.4s	remaining: 2m 25s
400:	test: 0.7679546	best: 0.7679546 (400)	total: 21.2s	remaining: 2m 17s
600:	test: 0.7718383	best: 0.7718383 (600)	total: 32.2s	remaining: 2m 8s
800:	test: 0.7737490	best: 0.7737490 (800)	total: 43.6s	remaining: 1m 59s
1000:	test: 0.7747734	best: 0.7747792 (999)	total: 54.9s	remaining: 1m 49s
1200:	test: 0.7757699	best: 0.7757768 (1199)	total: 1m 5s	remaining: 1m 38s
1400:	test: 0.7763218	best: 0.7763340 (1399)	total: 1m 16s	remaining: 1m 27s
1600:	test: 0.7765489	best: 0.7765489 (1600)	total: 1m 28s	remaining: 1m 17s
1800:	test: 0.7767429	best: 0.7767755 (1787)	total: 1m 39s	remaining: 1m 6s
2000:	test: 0.7767453	best: 0.7768467 (1890)	total: 1m 50s	remaining: 55.2s
2200:	test: 0.7768921	best: 0.7769771 (2180)	total: 2m 1s	remaining: 44.1s
2400:	test: 0.7769591	best: 0.7770145 (2372)	total: 2m 12s	remaining: 33.2s
2600:	test: 0.7769816	best

### Remaning Redundant Correlated Features